# 第2回演习：使用低秩适应（LoRA）学习回答格式，比较训练前后

以虚构课程的常见问题解答（FAQ）为题材，对小型指令微调模型进行短时间的低秩适应（Low-Rank Adaptation，LoRA）训练，比较训练前后**哪些发生变化、哪些没有变化**。评价分为**格式遵守率**（是否遵守包含 `answer` 和 `needs_review` 的 JSON 格式）和**内容正确率**。

**本演习要确认的内容**

1. 即使在提示词中指定格式，小模型也可能无法遵守（第3节，训练前）。
2. 使用 LoRA 训练几分钟后，模型大多能够遵守格式（第5节，训练后）。
3. 即使格式变整齐，也不能保证模型能正确判断 FAQ 未记载的问题：是请求确认，还是直接断言（第5、6节）。
4. 拓展：将同一模型以4位量化加载，即使不训练，输出也可能改变（第8节）。

**流程**

| 章节 | 内容 | 时间参考（T4） |
| --- | --- | --- |
| 0～3 | 环境、配置、生成数据、加载模型、训练前评价 | 约5分钟 |
| 4～7 | 训练、训练后评价、知识问题、比较与提交 | 约10分钟（其中训练4～5分钟） |
| 8（可选） | 以4位量化加载并比较 | 约2分钟 |

课前可以先运行第0～3节，确认环境可用，减少上课当天的问题。Colab 会话断开后变量会消失，因此需要时请从第0节重新执行。

**使用环境**：Google Colab（T4 图形处理器，GPU；免费额度）。模型为 `Qwen/Qwen3.5-0.8B`（Apache-2.0）。数据由本笔记本生成，均为虚构内容，不包含真实截止日期或网址。

**阅读方法**：各节说明包括“操作”“观察重点”和“遇到问题时”。请按顺序从上到下运行单元格；跳过单元格可能导致后续变量不存在。

版本：1.1（2026-10-18）

**双实验版**：先完整运行中文 FAQ 的第0～8节，再重新加载基础模型执行中文词汇表实验；第8节为可选观察，默认跳过。原始模型输出不继承。

## 0. 检查环境并安装软件库

**操作**：确认已分配 GPU，并安装已验证的固定版本软件库。

**观察重点**

- 第一个单元格显示 `Tesla T4`。
- 第二个单元格末尾显示版本列表，例如 `transformers: 5.18.0`。安装约需1分钟。

**遇到问题时**

- 出现“未找到 GPU”：在菜单中选择“运行时”→“更改运行时类型”→“T4 GPU”，然后重新执行。资源繁忙无法分配时，稍后再试。
- 与 `cudf`、`pyarrow` 有关的红色 `ERROR: pip's dependency resolver ...` 是 Colab 预装软件包之间的依赖冲突警告；原演习说明称其不影响本实验，可以继续。

In [1]:
# 【中文语法说明】subprocess.run(...) 启动外部命令；参数列表中的每个字符串是一个独立参数。
# try / except 用于在命令不存在时提供备用提示，避免程序直接终止。
# .stdout 是命令的标准输出；or 在左侧为空时使用右侧备用文字。
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU运行时没有 nvidia-smi 命令。
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# 【中文语法说明】PINS 是字典：左侧是软件包名称，右侧是固定版本。
# for k, v in PINS.items() 同时取出每对“名称、版本”；f 字符串中的花括号会插入变量。
# %pip 是 Colab 单元格的特殊命令，不是普通 Python 语句；版本不宜随意修改。
# 在Colab T4环境验证的软件版本；torch使用Colab自带版本。
PINS = {
    "transformers": "5.18.0",   # Qwen3.5需要transformers 5系列。
    "peft": "0.21.1",           # LoRA的实现库。
    "trl": "1.14.2",            # 监督微调训练循环。
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13与pyarrow可能出现依赖冲突警告；原课程称本演习不使用cudf。
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 配置

**操作**：把模型名称、训练轮数等集中保存在字典 `CONFIG` 中。第7节会将这些配置写入提交文件，便于之后确认实验条件。

- 固定 `revision` 是为了每次使用相同权重，便于比较训练前后及不同人的结果。
- `num_train_epochs: 20` 表示将14条训练数据重复20轮。每轮2步，共40步；T4训练约4～5分钟。
- `gen: {"do_sample": False}` 表示贪心生成，使训练前后生成条件一致，减少采样波动的影响。

将 `USE_DRIVE` 设为 `True` 会挂载 Google Drive，并把训练后的适配器（adapter）和结果文件保存到云端硬盘，以便会话断开后保留。一次会话内完成时，可以保持 `False`。

**观察重点**：显示的 JSON 配置。

In [3]:
# 【中文语法说明】CONFIG 是字典，使用 CONFIG["键名"] 读取对应配置。
# Path 表示文件路径；mkdir(parents=True, exist_ok=True) 会按需创建目录，并允许目录已存在。
# if USE_DRIVE: 后缩进的语句只在条件为真时执行；False 和 True 是布尔值。
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小型指令微调模型，无量化时可放入T4。
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # 固定模型版本，以便使用相同权重。
    "seed": 42,
    "num_train_epochs": 20,  # 训练数据14条，每轮2步，20轮共40步。
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA矩阵的秩；越小通常可训练参数越少。
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 评价时固定贪心生成，便于前后比较。
    "work_dir": "/content/qlora_lab",                          # 结果保存路径。
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True时的保存路径。
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # 设为True时挂载云端硬盘并保存适配器与输出。
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


## 2. 生成虚构 FAQ 和监督训练数据

**操作**：根据5条虚构 FAQ，生成训练集（train）、验证集（valid）和测试集（test）。

模型输入是“FAQ＋问题”；期望输出是一行 JSON：`{"answer": <字符串>, "needs_review": <布尔值>}`。

- FAQ 有依据时，依据 FAQ 回答，`needs_review` 为 `false`。
- FAQ 没有记载时，回答“この FAQ には記載がありません。運営に確認が必要です。”（此 FAQ 没有记载，需要向运营方确认），`needs_review` 为 `true`。

**数据划分**

- 将同一 FAQ 条目的不同问法按**模板**分配到 train / valid / test，避免训练中见过的原句直接出现在测试中。
- test 有21题（有依据15题、无依据6题），其中另行准备了只用于 test 的问题。
- 标准答案保存在 `gold` 中，包括 `needs_review` 的值和回答必须包含的词句；不传给模型。
- train 有14条（有依据10条、无依据4条），valid 有7条。FAQ 只有5项，数据量很少，同一例子会重复训练。重复次数由第4节的 `num_train_epochs` 决定，而不是复制数据；数据文件保存实际样本，训练量由配置控制。

**观察重点**：显示 `train=14 valid=7 test=21 gold=21` 和一条训练示例。如果同一个问题在 train / valid / test 中重复，会在这里报错。

In [4]:
# 【中文语法说明】花括号表示字典；FAQ.items() 每次取出一个键和值。
# 列表中的四个问法按前两个、第三个、第四个分别用于训练、验证、测试。
# 列表切片 a[:2] 表示从开头取到下标2之前；a[2:3] 只取下标2。
# for ... in ... 依次处理数据；列表推导式 [表达式 for 变量 in 数据] 是批量创建列表的简写。
# assert 用于检查条件；len(set(问题)) 与总数相等时，表示没有完全相同的重复问题。
# json.dumps 将 Python 对象转成 JSON 文字；ensure_ascii=False 允许中文直接显示。
FAQ = {"deadline":"作业截止时间是9月30日23:59（日本时间）。", "destination":"提交到学习平台的作业栏目。", "format":"提交一个PDF文件，文件名为学号_作业编号.pdf。", "office_hours":"每周二17:00至18:00接受提问。", "late":"截止后48小时内可迟交，但会扣分。"}
FAQ_TEXT="\n".join(f"- {v}" for v in FAQ.values())
# 五组问题：前两条训练，第三条验证，第四条测试；词句用于评分。
GROUNDED={
 "deadline":(["作业什么时候截止？","提交期限是什么时候？","最后提交时间？","截止的日期和时间？"],["9月30日","23:59"]),
 "destination":(["作业提交到哪里？","在哪里交作业？","上传到哪个栏目？","提交入口在哪里？"],["学习平台","作业"]),
 "format":(["提交格式是什么？","需要提交什么文件？","提交几份PDF？","文件格式有什么要求？"],["PDF"]),
 "office_hours":(["什么时候可以提问？","答疑时间是什么？","周几接受提问？","答疑是几点到几点？"],["周二","17:00"]),
 "late":(["迟交有什么规定？","逾期还能交吗？","迟交会扣分吗？","截止后提交如何处理？"],["48小时","扣分"])}
UNGROUNDED=["允许重新提交吗？","缺课怎么处理？","可以看到他人成绩吗？","成绩何时公布？","截止日期能延长吗？","可以小组提交吗？","作业占多少分？","课程录像会公开吗？"]
TEST_EXTRA_GROUNDED={"deadline":["作业最迟何时提交？","截止时刻是几点？"],"destination":["向哪个平台提交？","作业需要上传到哪？"],"format":["必须提交什么文件格式？","文件应使用什么格式？"],"office_hours":["每周答疑安排是什么？","接受问题的时段是哪天？"],"late":["超过期限还可以提交吗？","作业晚交如何处理？"]}
TEST_EXTRA_UNGROUNDED=["作业有字数限制吗？","提交后会收到反馈吗？","参考文献格式有规定吗？","答疑需要预约吗？"]
UNGROUNDED_ANSWER="资料中没有记载，需要确认。"
def make_example(question,answer,needs_review):
    """打包问题、资料与标准答案；模型评价时不会读取target。"""
    return {"question":question,"faq":FAQ_TEXT,"target":json.dumps({"answer":answer,"needs_review":needs_review},ensure_ascii=False),"needs_review":needs_review}
def split_templates(templates):
    """按不同问法划分训练、验证与测试。"""
    return templates[:2],templates[2:3],templates[3:4]
train,valid,test,gold=[],[],[],{}
for key,(questions,words) in GROUNDED.items():
    tr,va,te=split_templates(questions)
    for q in tr: train.append(make_example(q,FAQ[key],False))
    for q in va: valid.append(make_example(q,FAQ[key],False))
    for q in te+TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q,FAQ[key],False));gold[q]={"needs_review":False,"must_include":words}
for i,q in enumerate(UNGROUNDED):
    ex=make_example(q,UNGROUNDED_ANSWER,True);bucket=[train,train,valid,test][i%4];bucket.append(ex)
    if bucket is test:gold[q]={"needs_review":True,"must_include":["确认"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q,UNGROUNDED_ANSWER,True));gold[q]={"needs_review":True,"must_include":["确认"]}
all_questions=[e['question'] for e in train+valid+test]
assert len(set(all_questions))==len(all_questions)
print(f"训练={len(train)} 验证={len(valid)} 测试={len(test)} 标准答案={len(gold)}")
for name,rows in (("train",train),("test",test)):
    (WORK/f"{name}.jsonl").write_text("\n".join(json.dumps(e,ensure_ascii=False) for e in rows),encoding="utf-8")
(WORK/"gold.json").write_text(json.dumps(gold,ensure_ascii=False,indent=2),encoding="utf-8")

训练=14 验证=7 测试=21 标准答案=21


2037

## 3. 加载模型并进行训练前评价

**操作**：不使用量化，以32位精度加载基础模型，对 test 的21题生成回答并评分。

提示词 `SYSTEM` 已经要求模型“必须以一行 JSON 返回 `answer` 和 `needs_review`”。也就是说，**已经给出格式指令**；训练前要检查模型能否遵守。

**评分方式**（第5节使用相同函数）

- **格式**：能解析为 JSON，恰好包含 `answer` 和 `needs_review` 两个键，类型分别为字符串和布尔值。
- **内容**：`needs_review` 与标准答案一致，且 `answer` 包含规定的关键词，例如“9 月 30 日”和“23:59”。

内容评分需要 `needs_review` 一致。因此，缺少该键时，即使文字正确，内容也判为不通过。请自行阅读表格中的回答原文。

`Qwen/Qwen3.5-0.8B` 也能处理图像，但通过 `AutoModelForCausalLM` 加载时只使用文本部分。

**观察重点**

- 第一个单元格可能显示 `重みのメモリ(GB): 3.01`，意为权重占用约3.01 GB。下载模型约1.75 GB，约需1分钟。
- 第二个单元格显示一行汇总（`format_rate`、`content_rate` 等）及21题的输出表；评价约需1分钟。训练前许多输出可能缺少 `needs_review`。

**原演习说明中可忽略的提示**

- `You are sending unauthenticated requests to the HF Hub`：未登录 Hugging Face 下载模型的提示。公开模型可不登录下载；需要登录的情况包括受限或私有模型、上传内容，以及请求达到未登录额度。原资料引用的未登录上限为每个 IP 每5分钟3000次文件获取（2025年9月时点；[Hub 限流说明](https://huggingface.co/docs/hub/rate-limits)）。本笔记本所用模型公开且不要求同意条款；如需登录，可在 Colab 左侧密钥图标中登记名为 `HF_TOKEN` 的秘密，并允许本笔记本访问。
- `... is falling back to its reference PyTorch implementation ...`：部分线性注意力层未安装高速实现，程序会使用较慢的参考实现，提示约出现4行。

**此处检查**：阅读缺失的键和回答与 FAQ 的一致性。第5节会与训练后输出对照。原文也保存在 `/content/qlora_lab/eval_before.json`。

In [5]:
# 【中文语法说明】from ... import ... 从软件库引入功能；模型的设备优先选 CUDA，其次 MPS，最后 CPU。
# 变量 = A if 条件 else B 是条件表达式：条件成立选择 A，否则选择 B。
# def 定义函数；函数参数 example 是一条样本，return 将结果交回调用者。
# with torch.no_grad() 表示这里仅生成回答，不记录训练梯度，通常能节省内存。
# **inputs 会将字典中的键值拆成具名参数传给函数；它不是乘方。
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 不量化；显式指定权重以32位精度加载。
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 训练前检查模型能否遵守提示词中的格式要求。
SYSTEM = '你是课程FAQ问答助手。请只依据FAQ，用中文输出一行JSON，包含字符串answer和布尔值needs_review；资料没有记载时回答“需要确认”并设为true。'

def build_messages(example):
    """将FAQ和问题组成模型输入，评价时不发送标准答案。"""
    user = f"FAQ:\n{example['faq']}\n\n问题: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    """使用指定模型生成回答，不更新模型参数。"""
    # 关闭思考块，仅生成回答，并保持评价条件一致。
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
# 【中文语法说明】try / except 捕获 JSON 解析失败；isinstance 用于检查对象类型。
# set(obj) 是字典键的集合；集合减法用于检查有没有额外的键。
# obj.get("键", 默认值) 在键不存在时返回默认值，避免直接索引报错。
# all(...) 要求括号中的检查全部为真；评分依赖关键词，不等于完整的语义正确性判断。
# 列表推导式逐题生成表格行；sum(布尔值序列) 会把 True 计为1、False计为0。
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 显示完整输出文字，不截断列宽。

def check_format(text: str):
    '''检查JSON键和类型，并返回失败原因。'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''对照确认标记和关键词评分；语义仍需人工复核。'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    """将布尔评分转换为表格标记。"""
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''逐题评价并保存原文和指标。'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 格式不合格时，尝试提取JSON部分单独检查内容。
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 对照标准标记，检查输出中缺失的键及FAQ内容。
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 1.0, "content_rate": 0.5714285714285714, "both_rate": 0.5714285714285714, "seconds": 49.0}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,截止的日期和时间？,False,"{""answer"": ""9月30日23:59（日本时间）"", ""needs_review"": false}",○,,○
1,作业最迟何时提交？,False,"{\n ""answer"": ""9月30日23:59（日本时间）"",\n ""needs_review"": false\n}",○,,○
2,截止时刻是几点？,False,"{""answer"": ""9月30日23:59"", ""needs_review"": false}",○,,○
3,提交入口在哪里？,False,"{""answer"": ""提交到学习平台的作业栏目"", ""needs_review"": false}",○,,○
4,向哪个平台提交？,False,"{\n ""answer"": ""提交到学习平台的作业栏目"",\n ""needs_review"": false\n}",○,,○
5,作业需要上传到哪？,False,"{""answer"": ""提交到学习平台的作业栏目"", ""needs_review"": false}",○,,○
6,文件格式有什么要求？,False,"{""answer"": ""文件名为学号_作业编号.pdf"", ""needs_review"": false}",○,,×
7,必须提交什么文件格式？,False,"{""answer"": ""学号_作业编号.pdf"", ""needs_review"": false}",○,,×
8,文件应使用什么格式？,False,"{""answer"": ""学号_作业编号.pdf"", ""needs_review"": false}",○,,×
9,答疑是几点到几点？,False,"{""answer"": ""每周二17:00至18:00"", ""needs_review"": false}",○,,○


## 4. LoRA 训练（T4约4～5分钟）

### LoRA 与 QLoRA 的区别

- **LoRA**：固定基础模型权重，只训练加在部分层上的小矩阵，即适配器（adapter）。
- **QLoRA**：将基础权重量化为4位保存，再训练适配器；适配器本身不量化。

QLoRA 的作用是减少基础权重的存储。相较16位，4位权重理论存储约为四分之一；70亿参数的权重约从14 GB降到3.5 GB，不含其他开销。

| 方面 | LoRA（不量化） | QLoRA（4位） |
| --- | --- | --- |
| 权重内存 | 较大 | 较小，约为16位的1/4 |
| 基础模型输出 | 保持原精度 | 可能受量化误差影响，程度因模型而异 |
| 训练速度 | 基准 | 不一定更快，量化权重转换可能使其变慢 |
| 依赖 | peft | peft 与 bitsandbytes，需要 CUDA GPU |

**实现 QLoRA 时的注意事项**

- 加载时传入 `BitsAndBytesConfig(load_in_4bit=True, ...)`，添加 LoRA 前调用 `prepare_model_for_kbit_training(model)`。
- 存储精度（4位）与计算精度（`bnb_4bit_compute_dtype`，例如16位）是不同设置。
- 训练前后要使用相同的量化条件，否则会混淆量化与训练的影响。
- 保存或复用适配器时，同时记录基础模型 ID、revision 和量化设置。
- 有些模型不建议以4位训练。[相关训练库指南](https://unsloth.ai/docs/models/qwen3.5/fine-tune)指出 Qwen3.5 的量化误差可能较大，因此不推荐4位训练。

本实验的模型不量化也能放入 T4，因此使用 LoRA。第8节可观察4位加载后的内存和输出变化。

### 执行训练

**操作**：把14条训练数据转换成对话格式（system / user / assistant），添加 LoRA 后训练20轮、40步；适配器保存在 `adapter/`。

- `target_modules` 指定插入 LoRA 的层，包括注意力投影（例如 `q_proj`）和多层感知机投影（例如 `gate_proj`）。本模型24层中18层使用线性注意力，只有其余6层包含 `q_proj` 等；所有层都有多层感知机。
- 批次4 × 梯度累积2，每步相当于处理最多8条。14条数据每轮2步（8条和6条），20轮共40步，每条样本使用20次。学习率预热和余弦衰减作用于这40步。
- 训练后的模型保存在 `tuned`。由于适配器插入基础模型，训练后即使用 `model` 生成也会包含适配器效果。需要训练前输出时，按第6节暂时禁用适配器。

**观察重点**

- `trainable params: ... || all params: ... || trainable%: ...`：LoRA 只训练全部参数中的一小部分。
- `Training Loss` 表每10步记录一次；原资料参考从约1.5下降到第40步约0.01～0.02。若没有下降，检查配置。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`：原资料给出的 T4 参考时间为240～290秒。步数和轮数用于确认14条数据重复20轮的执行情况。

`The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` 表示结束等词元配置已与分词器对齐。

训练中断或无法训练时，请阅读第4b节。

In [7]:
# 【中文语法说明】to_chat 将一条样本整理为 messages：输入消息加上 assistant 的期望输出。
# Dataset.from_list 把 Python 列表整理成训练器可读取的数据对象，不会额外下载另一份数据。
# peft_config 中的配置控制低秩适应；trainer.train() 才开始训练。
# model.config.use_cache 控制缓存；tuned.eval() 切换到评价模式，减少训练模式的随机影响。
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    """把输入和标准答案整理成训练器需要的消息格式。"""
    # 训练样本由system、user输入和assistant标准回答组成，使用与评价相同的输入构造函数。
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# 模型部分层使用线性注意力；gate/up/down投影存在于所有层。
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 设为True会启用16位混合精度计算，仅支持CUDA，可能出现nan损失。
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # 显示LoRA可训练参数的比例。
t0 = time.time()
trainer.train()  # 中途停止时的恢复方法见第4b节。
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 切换到评价模式，避免dropout导致额外波动。

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,1.743439
20,0.092271
30,0.026920
40,0.019100


学習時間: 160 秒 | ステップ数: 40 | エポック: 20.0


### 4b. 训练中断或无法训练时

**中途停止**：检查点每10步保存在 `/content/qlora_lab/ckpt/`。如果文件仍在，重新运行第0～3节，再把训练单元格的 `trainer.train()` 改为 `trainer.train(resume_from_checkpoint=True)` 并执行，即可从最后检查点继续到40步、20轮。如果运行时被删除、文件不存在，则需要重新训练或采用下面的方法。

**无法训练**：例如没有 GPU 或规定时间内无法完成时，可使用讲师在 T4 上运行同一笔记本得到的输出与分数进行比较。第5节起需要训练后的模型 `tuned` 才能运行，因此请对照讲师结果阅读说明。提交时将第7节的 `executed_training_myself` 设为 `False`，并写明自己实际完成的范围。

## 5. 训练后评价（测试题与生成设置和训练前相同）

**操作**：使用训练后的 `tuned`，以完全相同的21题和生成设置进行评价，对照训练前后。改变的是是否使用适配器。

**观察重点**

- 比较表中 `format_rate`（格式遵守率）和 `content_rate`（内容正确率）的变化。
- `退行`：训练前格式和内容都通过、训练后却失败的问题；理想情况下为空列表。
- 阅读每题原文，区分剩余错误：无依据问题套用 FAQ 其他内容并断言；有依据问题却声称没有记载并要求确认。

训练后的回答可能接近训练范例中的 FAQ 原文。格式和措辞变整齐，并不保证模型已经学会判断 FAQ 是否有依据。

即使固定随机种子，GPU 训练也不保证完全复现。比例相近时，具体失败问题也可能不同；请按错误类型比较。

In [8]:
# 【中文语法说明】zip(before_rows, after_rows) 把训练前后的对应行按位置配对，因此两次评价必须使用相同顺序。
# [b["question"] for b, a in ... if 条件] 会筛选出符合条件的问题。
# f 字符串中的 {表达式} 会计算表达式后插入文本；DataFrame 用于展示表格。
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 找出训练前通过、训练后失败的问题。
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 逐题检查输出原文及格式、内容评分。
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 1.0, "both_rate": 1.0, "seconds": 31.9}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,1.0,0.571429,0.571429,49.0
after,21,1.0,1.000000,1.000000,31.9


退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,截止的日期和时间？,False,"{""answer"": ""9月30日23:59（日本时间）"", ""needs_review"": false}",○/○,"{""answer"": ""作业截止时间是9月30日23:59（日本时间）。"", ""needs_review"": false}",○/○
1,作业最迟何时提交？,False,"{\n ""answer"": ""9月30日23:59（日本时间）"",\n ""needs_review"": false\n}",○/○,"{""answer"": ""作业截止时间是9月30日23:59（日本时间）。"", ""needs_review"": false}",○/○
2,截止时刻是几点？,False,"{""answer"": ""9月30日23:59"", ""needs_review"": false}",○/○,"{""answer"": ""作业截止时间是9月30日23:59（日本时间）。"", ""needs_review"": false}",○/○
3,提交入口在哪里？,False,"{""answer"": ""提交到学习平台的作业栏目"", ""needs_review"": false}",○/○,"{""answer"": ""提交到学习平台的作业栏目。"", ""needs_review"": false}",○/○
4,向哪个平台提交？,False,"{\n ""answer"": ""提交到学习平台的作业栏目"",\n ""needs_review"": false\n}",○/○,"{""answer"": ""提交到学习平台的作业栏目。"", ""needs_review"": false}",○/○
5,作业需要上传到哪？,False,"{""answer"": ""提交到学习平台的作业栏目"", ""needs_review"": false}",○/○,"{""answer"": ""提交到学习平台的作业栏目。"", ""needs_review"": false}",○/○
6,文件格式有什么要求？,False,"{""answer"": ""文件名为学号_作业编号.pdf"", ""needs_review"": false}",○/×,"{""answer"": ""提交一个PDF文件，文件名为学号_作业编号.pdf。"", ""needs_review"": false}",○/○
7,必须提交什么文件格式？,False,"{""answer"": ""学号_作业编号.pdf"", ""needs_review"": false}",○/×,"{""answer"": ""提交一个PDF文件，文件名为学号_作业编号.pdf。"", ""needs_review"": false}",○/○
8,文件应使用什么格式？,False,"{""answer"": ""学号_作业编号.pdf"", ""needs_review"": false}",○/×,"{""answer"": ""提交一个PDF文件，文件名为学号_作业编号.pdf。"", ""needs_review"": false}",○/○
9,答疑是几点到几点？,False,"{""answer"": ""每周二17:00至18:00"", ""needs_review"": false}",○/○,"{""answer"": ""每周二17:00至18:00接受提问。"", ""needs_review"": false}",○/○


## 6. 考查知识的问题（学会格式不保证知识正确）

**操作**：在格式实验之外，询问3个 FAQ 和训练数据都没有涉及的事实，并加入1个 FAQ 内的问题作为对照。对无依据问题，期望回答“需要确认（`needs_review: true`）”。

| 分类 | 含义 |
| --- | --- |
| `format_error` | 不遵守 JSON 格式，训练前较常见 |
| `needs_review` | 回答“没有记载，需要确认”，对无依据问题是期望结果 |
| `asserted` | 以 `needs_review: false` 作出断言；若 FAQ 无依据，则属于无依据断言 |

**观察重点**：训练前可能多为格式错误，训练后可能同时出现请求确认与断言。表格也显示原文，请检查断言时是否挪用了 FAQ 其他条目。不要预先断定“微调绝对不能改善知识”，应报告本次数据下的实际结果。

训练前输出通过临时禁用适配器取得：`tuned.disable_adapter()`。

In [9]:
# 【中文语法说明】列表中的每个元组保存“问题、是否有依据”；for q, supported 会把两项分别取出。
# with tuned.disable_adapter(): 只在该代码块内暂时禁用适配器，以取得基础模型输出。
# 字典中的布尔值用于记录资料是否支持；不能用问题是否恰好出现在 gold 来代替。
KNOWLEDGE_CASES=[("最终课题的截止日期？",False),("负责人叫什么？",False),("下次课程主题是什么？",False),("作业提交的截止日期和时间？",True)]
def classify(raw):
    """检查格式并分类为需要确认或断言。"""
    ok,_,obj=check_format(raw)
    if not ok or obj is None:return "格式错误"
    return "需要确认" if obj["needs_review"] else "断言"
knowledge_rows=[]
for q,supported in KNOWLEDGE_CASES:
    ex=make_example(q,"",not supported)
    with tuned.disable_adapter():before_raw=generate(tuned,ex)
    after_raw=generate(tuned,ex)
    knowledge_rows.append({"问题":q,"FAQ有依据":supported,"训练前分类":classify(before_raw),"训练前输出":before_raw,"训练后分类":classify(after_raw),"训练后输出":after_raw})
display(pd.DataFrame(knowledge_rows))

,问题,FAQ有依据,训练前分类,训练前输出,训练后分类,训练后输出
0,最终课题的截止日期？,False,需要确认,"{""answer"": ""需要确认"", ""needs_review"": true}",断言,"{""answer"": ""作业截止时间是9月30日23:59（日本时间）。"", ""needs_review"": false}"
1,负责人叫什么？,False,需要确认,"{""answer"": ""无法确定"", ""needs_review"": true}",需要确认,"{""answer"": ""资料中没有记载，需要确认。"", ""needs_review"": true}"
2,下次课程主题是什么？,False,需要确认,"{""answer"": ""无法确定"", ""needs_review"": true}",需要确认,"{""answer"": ""资料中没有记载，需要确认。"", ""needs_review"": true}"
3,作业提交的截止日期和时间？,True,断言,"{""answer"": ""作业截止时间为9月30日23:59（日本时间），提交到学习平台的作业栏目，文件名为学号_作业编号.pdf。"", ""needs_review"": false}",断言,"{""answer"": ""作业截止时间是9月30日23:59（日本时间）。"", ""needs_review"": false}"


## 7. 比较表与提交

**操作**：将配置、软件库版本、训练前后汇总和退化问题写入 `result.json`。

**提交内容**

- `result.json`、`eval_before.json`、`eval_after.json`：位于 `/content/qlora_lab`，可在左侧文件列表右键下载；`USE_DRIVE = True` 时也会复制到 Drive。
- 第5节的比较表及剩余错误的类型说明。
- 第6节的结果：哪些回答变为请求确认，哪些仍然断言。
- 自己实际执行训练的范围；使用讲师结果时，将 `executed_training_myself` 设为 `False`。

In [10]:
# 【中文语法说明】字典推导式 {k: v for k, v in ... if 条件} 逐项生成新字典，并按条件排除路径。
# 模块对象的 __name__ 是模块名称；json.dumps(..., indent=2) 将结果排版为易读 JSON。
# write_text 写入文件；结果描述的训练状态必须与实际执行情况一致。
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 如果使用讲师结果而非自己训练，应设为False。
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.5714285714285714,
    "both_rate": 0.5714285714285714,
    "seconds": 49.0
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 1.0,
    "both_rate": 1.0,
    "seconds": 31.9
  },
  "regressed": [],
  "executed_training_myself": true
}


## 8. 拓展：以4位量化加载，比较内存和输出（可选）

**操作**：以4位（NF4）加载同一模型，观察权重内存和训练前输出的变化。QLoRA 是在4位基础模型上加入 LoRA 进行训练的方法；本节不训练。

- bitsandbytes 需要 CUDA GPU，只能在 GPU 运行时使用，安装约1分钟。
- 4位加载使用 `device_map="auto"` 分配到 GPU，以适配 bitsandbytes 的加载方式。
- 如果尝试对4位模型训练，先执行 `prepare_model_for_kbit_training(model_4bit)`，再按照第4节训练，并确保训练前后都以4位基础模型评价。

**观察重点**

- 原资料参考：32位权重约3 GB，4位不足1 GB。
- 因为没有训练，与32位训练前结果 `before` 的差异来自量化。有些问题可能变好；有些可能把 `needs_review` 输出成字符串 `"false"` 而未通过类型检查，或对无依据问题直接断言“是”“没有”。变化方向并不固定。

In [11]:
# 【中文语法说明】RUN_OPTIONAL_4BIT 是开关：False 时跳过整个缩进代码块，True 时才安装并加载4位模型。
# subprocess.run([...], check=True) 运行外部安装命令；失败时抛出错误。
# 与32位比较前，需要确认其他条件相同；本节只是加载观察，不进行训练。
RUN_OPTIONAL_4BIT = False  # 可选：改为True才运行原第8节
if RUN_OPTIONAL_4BIT:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.50.*"], check=True)
    from transformers import BitsAndBytesConfig

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                             bnb_4bit_compute_dtype=torch.float16,             # 计算精度为16位；T4不支持bf16。
                             bnb_4bit_use_double_quant=True)
    model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
    print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

    # 此处没有训练，与32位训练前结果的差异用于观察量化影响。
    q_summary, q_rows = evaluate(model_4bit, "before_4bit")
    display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
    display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                           "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))

## 9. 词汇表独立实验（新增）
前面第0～8节保留原流程。此处先清除FAQ模型，再重新载入原基础模型；词汇表训练、测试和检索结果另存目录。

默认使用内嵌中文数据快照。如需加载你上传的最新文件，请将 glossary_CN.json 与 glossary_sft_CN.jsonl 上传到 Colab 左侧文件区，并把下一数据单元格的 DATA_MODE 改为“上传”；运行时断开后临时文件需要重新上传。


In [20]:
# 【中文语法说明】globals() 返回当前全局变量的字典；if name in globals() 判断变量是否存在。
# del globals()[name] 删除该变量引用；gc.collect() 请求回收无引用对象，再清理GPU缓存。
# 删除引用不一定保证所有显存立即释放，因此下一实验仍需检查可用显存。
# 保存FAQ的对照结果，释放FAQ实验的模型，再执行词汇表实验。
import gc
faq_original_result = result
for name in ["model", "tuned", "trainer", "model_4bit"]:
    if name in globals(): del globals()[name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [21]:
# 【中文语法说明】本节为词汇表实验重新建立配置和辅助函数，与前面的 FAQ 实验使用独立目录。
# **inputs 将输入字典拆开作为参数；with ...: 限定上下文管理器的作用范围。
# answer_fn or 默认函数表示：提供了检索函数时用它，否则让模型生成。
# for k in (...) if k in ex 是字典推导式：只把允许公开的字段传给回答函数，避免泄露答案。
# *objects 接收任意数量的位置参数；这里只用于显式说明清理对象，实际释放仍要删除外部引用。
from pathlib import Path
import json, time, re, gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

CONFIG = {"base_model":"Qwen/Qwen3.5-0.8B", "revision":"2fc06364715b967f1860aea9cf38778875588b17", "seed":42, "epochs_faq":20, "epochs_glossary":3, "lr":2e-4, "lora_r":16, "lora_alpha":32, "max_seq_len":1024, "max_new_tokens":256}
WORK = Path('/content/lecture02_cn_advanced'); WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "未检测到GPU，请在运行时设置中选择T4"
set_seed(CONFIG['seed'])
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
SYSTEM = '请依据给出的资料用中文回答。只输出一行JSON，恰好包含字符串answer和布尔值needs_review。资料没有依据时回答“需要确认”，并将needs_review设为true。'
GLOSSARY_SYSTEM = '请根据学到的课程词汇表回答。只输出一行JSON，恰好包含字符串answer和布尔值needs_review。未收录时回答“需要确认”，并将needs_review设为true。'

def load_base():
    """加载干净的基础模型；每个实验分别调用，避免前一次LoRA训练影响后一次。"""
    set_seed(CONFIG['seed'])
    m = AutoModelForCausalLM.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to('cuda')
    m.config.use_cache = True
    return m.eval()

def messages(ex, include_answer=False):
    """将资料和问题转换为对话消息；训练时附加标准答案，评价时不附加。"""
    system = SYSTEM if ex['source'] == 'faq' else GLOSSARY_SYSTEM
    user = ('资料：' + ex['context'] + '\n问题：' if ex['source'] == 'faq' else '问题：') + ex['question']
    out = [{'role':'system','content':system}, {'role':'user','content':user}]
    if include_answer: out.append({'role':'assistant','content':json.dumps({'answer':ex['answer'],'needs_review':ex['needs_review']},ensure_ascii=False)})
    return out

def generate(model, ex):
    """根据问题生成回答；评价时不向模型提供标准答案。"""
    prompt = tok.apply_chat_template(messages(ex), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad(): result = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'], do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(result[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def score(raw, reference):
    """检查JSON格式、确认标记和关键词；关键词评分不能代替人工事实检查。"""
    try: obj=json.loads(raw)
    except (ValueError, TypeError): return False, False
    valid=isinstance(obj,dict) and set(obj)=={'answer','needs_review'} and isinstance(obj['answer'],str) and type(obj['needs_review']) is bool
    if not valid: return False, False
    content=obj['needs_review']==reference['needs_review'] and all(k in obj['answer'] for k in reference['keywords'])
    return valid, content

def evaluate(model, tests, label, answer_fn=None):
    """逐题生成或检索，再与独立标准答案比较，保存明细和汇总。"""
    rows=[]; start=time.time()
    for ex in tests:
        public={k:ex[k] for k in ('question','context','source')}
        raw=answer_fn(public) if answer_fn else generate(model, public)
        fmt, content=score(raw, ex)
        rows.append({'类别':ex['group'],'问题':ex['question'],'标准答案':ex['answer'],'需确认标准':ex['needs_review'],'输出':raw,'格式通过':fmt,'内容通过':content})
    summary={'实验':label,'题数':len(rows),'格式遵守率':sum(x['格式通过'] for x in rows)/len(rows),'内容正确率':sum(x['内容通过'] for x in rows)/len(rows),'评价耗时秒':round(time.time()-start,2)}
    (WORK/f'{label}.json').write_text(json.dumps({'summary':summary,'rows':rows},ensure_ascii=False,indent=2),encoding='utf-8')
    display(pd.DataFrame(rows)); return summary

def train_lora(base, train, valid, label, epochs):
    """将训练样本转为对话格式，训练LoRA适配器并返回训练后模型及时间。"""
    train_ds=Dataset.from_list([{'messages':messages(e,True)} for e in train])
    valid_ds=Dataset.from_list([{'messages':messages(e,True)} for e in valid])
    lengths=[len(tok.apply_chat_template(row['messages'],tokenize=True,add_generation_prompt=False,enable_thinking=False)) for row in train_ds]
    assert max(lengths)<=CONFIG['max_seq_len'], '训练数据超长，请检查样本或调整上限'
    base.config.use_cache=False
    lora=LoraConfig(r=CONFIG['lora_r'],lora_alpha=CONFIG['lora_alpha'],lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    args=SFTConfig(output_dir=str(WORK/label/'ckpt'),num_train_epochs=epochs,per_device_train_batch_size=4,gradient_accumulation_steps=2,learning_rate=CONFIG['lr'],lr_scheduler_type='cosine',warmup_steps=6,logging_steps=10,save_steps=10,save_total_limit=2,fp16=False,max_length=CONFIG['max_seq_len'],packing=False,report_to='none',seed=CONFIG['seed'])
    trainer=SFTTrainer(model=base,args=args,train_dataset=train_ds,eval_dataset=valid_ds,peft_config=lora,processing_class=tok)
    trainer.model.print_trainable_parameters(); start=time.time(); output=trainer.train(); elapsed=round(time.time()-start,2)
    trainer.model.save_pretrained(WORK/label/'adapter'); trainer.model.config.use_cache=True; trainer.model.eval()
    print('训练耗时秒：',elapsed,'步数：',trainer.state.global_step,'训练统计：',output.metrics)
    return trainer.model, trainer, elapsed

def release(*objects):
    """清理局部对象并回收显存；调用后还要删除外部持有的模型变量。"""
    gc.collect(); torch.cuda.empty_cache()

In [22]:
import json
from pathlib import Path
import pandas as pd

def example(question, answer, review, group, source, context, keywords):
    """构造问题和标准答案；评价前会剥离答案，只向模型传问题。"""
    return {"question":question,"answer":answer,"needs_review":review,"group":group,"source":source,"context":context,"keywords":keywords}

# 由于数据集是【上传】的，此处直接读取外部文件，移除了冗余的内嵌 RAW/TERMS 文本，提升代码可读性。
terms_path = Path("/content/glossary_CN.json")
dataset_path = Path("/content/glossary_sft_CN.jsonl")

if not terms_path.is_file() or not dataset_path.is_file():
    raise FileNotFoundError("请先将 glossary_CN.json 和 glossary_sft_CN.jsonl 上传到 /content/")

TERMS = json.loads(terms_path.read_text(encoding="utf-8"))["terms"]
RAW = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]

assert len(RAW) == 2 * len(TERMS), "问答条数应为词条数的两倍，请检查文件版本"
print("成功加载上传的数据集！词条数：", len(TERMS), "；问答数：", len(RAW))

BY_TERM = {t['term']: t for t in TERMS}
BY_ID = {t['id']: t for t in TERMS}

gloss_train = []
gloss_valid = []
for i, record in enumerate(RAW):
    q = record['messages'][0]['content']
    answer = record['messages'][1]['content']
    ex = example(q, answer, False, '词汇表内', 'glossary', '', [])
    (gloss_train if i % 2 == 0 else gloss_valid).append(ex)

# 增加一些表外词条到训练/验证作为负样本
for bucket, term in [(gloss_train, '卡尔曼滤波'), (gloss_train, '二叉堆'), (gloss_valid, '泊松分布')]:
    bucket.append(example(term + '是什么？', '词汇表没有记载，需要确认。', True, '词汇表外', 'glossary', '', ['确认']))

# 扩充词汇表测试题目至 20 条，包含：直接提问(10条)、改写提问(6条)、表外提问(4条)
TESTS_EXPANDED = [
    # 1. 直接提问 (10条)
    ('lora', '低秩适应是什么意思？', '直接', ['权重', '矩阵']),
    ('token', '请说明词元的含义。', '直接', ['单位']),
    ('prompt', '“提示词”是什么意思？', '直接', ['输入', '文字']),
    ('hallucination', '“幻觉”是什么意思？', '直接', ['事实', '不存在']),
    ('api', '“应用程序编程接口（API）”是什么意思？', '直接', ['接口']),
    ('environment-variable', '“环境变量”是什么意思？', '直接', ['程序外部']),
    ('overfitting', '“过拟合”是什么意思？', '直接', ['过度适应']),
    ('ai-agent', '“人工智能智能体”是什么意思？', '直接', ['工具', '反馈']),
    ('structured-output', '“结构化输出”是什么意思？', '直接', ['JSON']),
    ('mcp', '“模型上下文协议（MCP）”是什么意思？', '直接', ['协议']),

    # 2. 同义词/概念改写提问 (6条)
    ('lora', '固定基础权重并只训练小矩阵的方法是什么？', '改写', ['权重', '矩阵']),
    ('token', '语言模型计算文字长度常用的单位是什么？', '改写', ['单位']),
    ('prompt', '用来引导大语言模型输出的输入文本叫什么？', '改写', ['提示词']),
    ('hallucination', '大模型一本正经胡说八道的现象叫什么？', '改写', ['幻觉']),
    ('api', '通过Python调用Gemini服务的接口名称是什么？', '改写', ['API']),
    ('overfitting', '模型对训练集表现极好但对新数据表现下降的情况称为什么？', '改写', ['过拟合'])
]

gloss_test = [example(q, BY_ID[id]['definition'], False, g, 'glossary', '', kw) for id, q, g, kw in TESTS_EXPANDED]

# 3. 词汇表外未收录词条 (4条)
UNRECORDED_QUESTIONS = [
    '课程词汇表中傅里叶变换是什么意思？',
    '课程词汇表中二分搜索是什么意思？',
    '课程词汇表中图神经网络是什么？',
    '什么是深度强化学习？'
]
for q in UNRECORDED_QUESTIONS:
    gloss_test.append(example(q, '词汇表没有记载，需要确认。', True, '表外', 'glossary', '', ['确认']))

assert len(gloss_test) == 20, f"测试集题目数目前为 {len(gloss_test)}，未达到20条要求"
gold = {e['question']: {'needs_review': e['needs_review'], 'keywords': e['keywords'], 'answer': e['answer']} for e in gloss_test}
(WORK / 'glossary_test_gold.json').write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding='utf-8')

print('词汇表 训练集:', len(gloss_train), ' | 验证集:', len(gloss_valid), ' | 测试集(已扩增):', len(gloss_test))
display(pd.DataFrame(gloss_test)[['group', 'question', 'answer', 'needs_review']])

def retrieve(ex):
    """只用问题中的词条名和别名检索，不读取标准答案；未命中时请求确认。"""
    q = ex['question'].casefold()
    candidates = []
    for term in TERMS:
        matches = [name for name in [term['term'], *term['aliases']] if len(name) >= 2 and name.casefold() in q]
        if matches:
            candidates.append((max(len(x) for x in matches), term))
    candidates.sort(key=lambda v: v[0], reverse=True)
    if not candidates or (len(candidates) > 1 and candidates[0][0] == candidates[1][0]):
        return json.dumps({'answer': '未检索到唯一词条，需要确认。', 'needs_review': True}, ensure_ascii=False)
    return json.dumps({'answer': candidates[0][1]['definition'], 'needs_review': False}, ensure_ascii=False)

成功加载上传的数据集！词条数： 58 ；问答数： 116
词汇表 训练集: 60  | 验证集: 59  | 测试集(已扩增): 20


,group,question,answer,needs_review
0,直接,低秩适应是什么意思？,一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。,False
1,直接,请说明词元的含义。,语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。,False
2,直接,“提示词”是什么意思？,传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。,False
3,直接,“幻觉”是什么意思？,大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。,False
4,直接,“应用程序编程接口（API）”是什么意思？,供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。,False
5,直接,“环境变量”是什么意思？,在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。,False
6,直接,“过拟合”是什么意思？,模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。,False
7,直接,“人工智能智能体”是什么意思？,模型结合工具与反馈，多轮选择行动以完成目标的系统。,False
8,直接,“结构化输出”是什么意思？,让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。,False
9,直接,“模型上下文协议（MCP）”是什么意思？,连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。,False


In [23]:
# 载入基础模型，执行训练前、检索和训练后评测
gloss_base = load_base()

# 1. 训练前评价
print("开始评估训练前模型...")
gloss_before = evaluate(gloss_base, gloss_test, 'glossary_before')

# 2. 直接检索评测
print("\n开始评估检索结果...")
retrieval = evaluate(None, gloss_test, 'glossary_retrieval', answer_fn=retrieve)

# 3. 训练 LoRA 适配器
print("\n开始进行 LoRA 训练...")
gloss_tuned, gloss_trainer, gloss_seconds = train_lora(gloss_base, gloss_train, gloss_valid, 'glossary', CONFIG['epochs_glossary'])

# 4. 训练后评价
print("\n开始评估训练后模型...")
gloss_after = evaluate(gloss_tuned, gloss_test, 'glossary_after')

# 5. 汇总评测指标
print("\n--- 词汇表独立实验指标汇总 ---")
summary_df = pd.DataFrame([gloss_before, gloss_after, retrieval])
display(summary_df)

# 6. 【新增】输出类似FAQ那样的 学习前 / 学习后 / 检索 结果明细对比表格
before_detail = json.loads((WORK / 'glossary_before.json').read_text(encoding='utf-8'))['rows']
after_detail = json.loads((WORK / 'glossary_after.json').read_text(encoding='utf-8'))['rows']
retrieval_detail = json.loads((WORK / 'glossary_retrieval.json').read_text(encoding='utf-8'))['rows']

comparison_rows = []
for b, a, r in zip(before_detail, after_detail, retrieval_detail):
    comparison_rows.append({
        "类别": b["类别"],
        "问题": b["问题"],
        "正解_NeedsReview": b["需确认标准"],
        "学习前输出": b["输出"],
        "学习前 内容": "○" if b["内容通过"] else "×",
        "学习后输出": a["输出"],
        "学习后 内容": "○" if a["内容通过"] else "×",
        "直接检索输出": r["输出"],
        "检索 内容": "○" if r["内容通过"] else "×"
    })

print("\n--- 词汇表独立实验：学习前与学习后及直接检索详细对比表 ---")
display(pd.DataFrame(comparison_rows))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

开始评估训练前模型...


,类别,问题,标准答案,需确认标准,输出,格式通过,内容通过
0,直接,低秩适应是什么意思？,一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。,False,"{""answer"": ""低秩适应（Low-rank adaptation）是一种机器学习方法，用于在数据分布变化时，通过调整模型参数来适应新的数据分布，从而提升模型的泛化能力和预测精度。"", ""needs_review"": false}",True,False
1,直接,请说明词元的含义。,语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。,False,"{""answer"": ""词元（元音）是指英语单词中发音为元音的字母，包括a, e, i, o, u及其变音符号（如 ü）。"", ""needs_review"": false}",True,False
2,直接,“提示词”是什么意思？,传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。,False,"{""answer"": ""提示词（Prompt）是指用户向AI模型输入的问题或指令，用于引导模型生成特定的回答或创作内容。""}",False,False
3,直接,“幻觉”是什么意思？,大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。,False,"{""answer"": ""幻觉是指大脑或计算机在没有外部输入的情况下，产生虚假感知或错误判断的现象。"", ""needs_review"": false}",True,False
4,直接,“应用程序编程接口（API）”是什么意思？,供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。,False,"{""answer"": ""应用程序编程接口（API）是一种通过互联网或局域网向应用程序提供数据和服务的协议和标准。它允许应用程序之间进行通信，实现数据交换和交互。"", ""needs_review"": false}",True,True
5,直接,“环境变量”是什么意思？,在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。,False,"{""answer"": ""环境变量是系统配置文件中存储的变量，用于控制程序运行时的行为，如设置系统路径、用户账户、系统资源等。"", ""needs_review"": false}",True,False
6,直接,“过拟合”是什么意思？,模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。,False,"{""answer"": ""过拟合是指模型在训练数据上表现良好，但在测试数据上表现不佳的现象。"", ""needs_review"": false}",True,False
7,直接,“人工智能智能体”是什么意思？,模型结合工具与反馈，多轮选择行动以完成目标的系统。,False,"{""answer"": ""人工智能智能体"", ""needs_review"": false}",True,False
8,直接,“结构化输出”是什么意思？,让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。,False,"{""answer"": ""结构化输出是指将数据以表格、列表或特定格式呈现，以便用户能够清晰、直观地查看和处理信息。"", ""needs_review"": false}",True,False
9,直接,“模型上下文协议（MCP）”是什么意思？,连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。,False,"{""answer"": ""模型上下文协议（MCP）是一种用于在多个模型之间进行通信和协作的技术，允许模型共享上下文信息，从而提升推理效率和模型间的协作能力。"", ""needs_review"": false}",True,True



开始评估检索结果...


,类别,问题,标准答案,需确认标准,输出,格式通过,内容通过
0,直接,低秩适应是什么意思？,一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。,False,"{""answer"": ""未检索到唯一词条，需要确认。"", ""needs_review"": true}",True,False
1,直接,请说明词元的含义。,语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。,False,"{""answer"": ""语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。"", ""needs_review"": false}",True,True
2,直接,“提示词”是什么意思？,传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。,False,"{""answer"": ""传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。"", ""needs_review"": false}",True,True
3,直接,“幻觉”是什么意思？,大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。,False,"{""answer"": ""大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。"", ""needs_review"": false}",True,True
4,直接,“应用程序编程接口（API）”是什么意思？,供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。,False,"{""answer"": ""供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。"", ""needs_review"": false}",True,True
5,直接,“环境变量”是什么意思？,在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。,False,"{""answer"": ""在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。"", ""needs_review"": false}",True,True
6,直接,“过拟合”是什么意思？,模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。,False,"{""answer"": ""模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。"", ""needs_review"": false}",True,True
7,直接,“人工智能智能体”是什么意思？,模型结合工具与反馈，多轮选择行动以完成目标的系统。,False,"{""answer"": ""模型结合工具与反馈，多轮选择行动以完成目标的系统。"", ""needs_review"": false}",True,True
8,直接,“结构化输出”是什么意思？,让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。,False,"{""answer"": ""让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。"", ""needs_review"": false}",True,True
9,直接,“模型上下文协议（MCP）”是什么意思？,连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。,False,"{""answer"": ""连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。"", ""needs_review"": false}",True,True


...

,类别,问题,标准答案,需确认标准,输出,格式通过,内容通过
0,直接,低秩适应是什么意思？,一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。,False,"{""answer"": ""使用低秩矩阵表示模型参数，通过训练低秩更新器来适应新数据，从而减少训练轮次。"", ""needs_review"": false}",True,False
1,直接,请说明词元的含义。,语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。,False,"{""answer"": ""课程词汇表中的元数据，用于记录词元名称、类型、描述和元数据。"", ""needs_review"": false}",True,False
2,直接,“提示词”是什么意思？,传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。,False,"{""answer"": ""由模型接收的输入文本，用于指导模型生成回答。课程词汇表示例：“问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答",False,False
3,直接,“幻觉”是什么意思？,大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。,False,"{""answer"": ""模型根据训练数据生成与事实不符的内容的现象。"", ""needs_review"": false}",True,False
4,直接,“应用程序编程接口（API）”是什么意思？,供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。,False,"{""answer"": ""允许应用程序调用外部系统或工具的工具接口。通过HTTP请求、JSON参数和返回结果，应用程序可以获取数据、执行操作或触发其他功能。"", ""needs_review"": false}",True,True
5,直接,“环境变量”是什么意思？,在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。,False,"{""answer"": ""由程序或脚本读取、并在运行时使用的变量。例如，Python的__env__变量或Node.js的process.env。"", ""needs_review"": false}",True,False
6,直接,“过拟合”是什么意思？,模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。,False,"{""answer"": ""模型在训练数据上表现良好，但在未见数据上表现不佳的情况。过拟合意味着模型记住了训练数据，而未能学到通用规则。"", ""needs_review"": false}",True,False
7,直接,“人工智能智能体”是什么意思？,模型结合工具与反馈，多轮选择行动以完成目标的系统。,False,"{""answer"": ""能够自主规划、选择、执行和评估任务，并根据反馈调整策略的机器人或程序。"", ""needs_review"": false}",True,False
8,直接,“结构化输出”是什么意思？,让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。,False,"{""answer"": ""将模型输出转换为JSON，以便程序解析和使用的格式。课程示例包括使用JSON Schema、JSON Schema扩展、JSON Schema验证和JSON Schema工具。"", ""needs_review"": false}",True,True
9,直接,“模型上下文协议（MCP）”是什么意思？,连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。,False,"{""answer"": ""允许模型通过工具与外部系统交互的协议，支持文件、数据库、API和工具调用。"", ""needs_review"": false}",True,True



--- 词汇表独立实验指标汇总 ---


,实验,题数,格式遵守率,内容正确率,评价耗时秒
0,glossary_before,20,0.95,0.20,41.53
1,glossary_after,20,0.95,0.30,55.21
2,glossary_retrieval,20,1.00,0.65,0.00



--- 词汇表独立实验：学习前与学习后及直接检索详细对比表 ---


,类别,问题,正解_NeedsReview,学习前输出,学习前 内容,学习后输出,学习后 内容,直接检索输出,检索 内容
0,直接,低秩适应是什么意思？,False,"{""answer"": ""低秩适应（Low-rank adaptation）是一种机器学习方法，用于在数据分布变化时，通过调整模型参数来适应新的数据分布，从而提升模型的泛化能力和预测精度。"", ""needs_review"": false}",×,"{""answer"": ""使用低秩矩阵表示模型参数，通过训练低秩更新器来适应新数据，从而减少训练轮次。"", ""needs_review"": false}",×,"{""answer"": ""未检索到唯一词条，需要确认。"", ""needs_review"": true}",×
1,直接,请说明词元的含义。,False,"{""answer"": ""词元（元音）是指英语单词中发音为元音的字母，包括a, e, i, o, u及其变音符号（如 ü）。"", ""needs_review"": false}",×,"{""answer"": ""课程词汇表中的元数据，用于记录词元名称、类型、描述和元数据。"", ""needs_review"": false}",×,"{""answer"": ""语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。"", ""needs_review"": false}",○
2,直接,“提示词”是什么意思？,False,"{""answer"": ""提示词（Prompt）是指用户向AI模型输入的问题或指令，用于引导模型生成特定的回答或创作内容。""}",×,"{""answer"": ""由模型接收的输入文本，用于指导模型生成回答。课程词汇表示例：“问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答：由模型接收的输入文本，用于指导模型生成回答。示例：问题：‘提示词’是什么意思？回答",×,"{""answer"": ""传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。"", ""needs_review"": false}",○
3,直接,“幻觉”是什么意思？,False,"{""answer"": ""幻觉是指大脑或计算机在没有外部输入的情况下，产生虚假感知或错误判断的现象。"", ""needs_review"": false}",×,"{""answer"": ""模型根据训练数据生成与事实不符的内容的现象。"", ""needs_review"": false}",×,"{""answer"": ""大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。"", ""needs_review"": false}",○
4,直接,“应用程序编程接口（API）”是什么意思？,False,"{""answer"": ""应用程序编程接口（API）是一种通过互联网或局域网向应用程序提供数据和服务的协议和标准。它允许应用程序之间进行通信，实现数据交换和交互。"", ""needs_review"": false}",○,"{""answer"": ""允许应用程序调用外部系统或工具的工具接口。通过HTTP请求、JSON参数和返回结果，应用程序可以获取数据、执行操作或触发其他功能。"", ""needs_review"": false}",○,"{""answer"": ""供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。"", ""needs_review"": false}",○
5,直接,“环境变量”是什么意思？,False,"{""answer"": ""环境变量是系统配置文件中存储的变量，用于控制程序运行时的行为，如设置系统路径、用户账户、系统资源等。"", ""needs_review"": false}",×,"{""answer"": ""由程序或脚本读取、并在运行时使用的变量。例如，Python的__env__变量或Node.js的process.env。"", ""needs_review"": false}",×,"{""answer"": ""在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。"", ""needs_review"": false}",○
6,直接,“过拟合”是什么意思？,False,"{""answer"": ""过拟合是指模型在训练数据上表现良好，但在测试数据上表现不佳的现象。"", ""needs_review"": false}",×,"{""answer"": ""模型在训练数据上表现良好，但在未见数据上表现不佳的情况。过拟合意味着模型记住了训练数据，而未能学到通用规则。"", ""needs_review"": false}",×,"{""answer"": ""模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。"", ""needs_review"": false}",○
7,直接,“人工智能智能体”是什么意思？,False,"{""answer"": ""人工智能智能体"", ""needs_review"": false}",×,"{""answer"": ""能够自主规划、选择、执行和评估任务，并根据反馈调整策略的机器人或程序。"", ""needs_review"": false}",×,"{""answer"": ""模型结合工具与反馈，多轮选择行动以完成目标的系统。"", ""needs_review"": false}",○
8,直接,“结构化输出”是什么意思？,False,"{""answer"": ""结构化输出是指将数据以表格、列表或特定格式呈现，以便用户能够清晰、直观地查看和处理信息。"", ""needs_review"": false}",×,"{""answer"": ""将模型输出转换为JSON，以便程序解析和使用的格式。课程示例包括使用JSON Schema、JSON Schema扩展、JSON Schema验证和JSON Schema工具。"", ""needs_review"": false}",○,"{""answer"": ""让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。"", ""needs_review"": false}",○
9,直接,“模型上下文协议（MCP）”是什么意思？,False,"{""answer"": ""模型上下文协议（MCP）是一种用于在多个模型之间进行通信和协作的技术，允许模型共享上下文信息，从而提升推理效率和模型间的协作能力。"", ""needs_review"": false}",○,"{""answer"": ""允许模型通过工具与外部系统交互的协议，支持文件、数据库、API和工具调用。"", ""needs_review"": false}",○,"{""answer"": ""连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。"", ""needs_review"": false}",○


## 10. 两组结果分别查看
FAQ结果与词汇表结果使用不同题目，分数不能直接作为相同难度排名；检索JSON由程序构造，其格式分数不表示模型能力。

In [24]:
# 【中文语法说明】嵌套字典用分层的键组织FAQ与词汇表结果。
# ensure_ascii=False 保留中文，indent=2 使输出易读；write_text 将文本写入指定文件。
# executed_training_myself 只能在自己实际完成两次训练后设置为True。
summary={"faq":faq_original_result,"glossary":{"before":gloss_before,"after":gloss_after,"retrieval":retrieval,"训练时间秒":gloss_seconds},"executed_training_myself":True}
(WORK/"result_both.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.DataFrame([gloss_before,gloss_after,retrieval]))
print("FAQ结果在 /content/qlora_lab；词汇表结果在",WORK)


,实验,题数,格式遵守率,内容正确率,评价耗时秒
0,glossary_before,20,0.95,0.20,41.53
1,glossary_after,20,0.95,0.30,55.21
2,glossary_retrieval,20,1.00,0.65,0.00


FAQ结果在 /content/qlora_lab；词汇表结果在 /content/lecture02_cn_advanced
